# Reproducing Zhu et al. (2022) from dandiset 001193

**Paper.** Zhu Y, Xu M, Lu J, Hu J, Kwok VPY, Zhou Y, Yuan D, Wu B, Zhang J, Wu J, Tan LH (2022).
Distinct spatiotemporal patterns of syntactic and semantic processing in human inferior frontal
gyrus. *Nature Human Behaviour* 6, 1104-1111.
[doi:10.1038/s41562-022-01334-6](https://doi.org/10.1038/s41562-022-01334-6).

**Claims file.** `claims/001193/10.1038-s41562-022-01334-6.yaml`, written from the version of
record (main text, figure legends, Methods, reporting summary) and the Supplementary Information
PDF, and checked by a second reader. The paper links no code.

**Dandiset.** [DANDI:001193](https://dandiarchive.org/dandiset/001193/0.240921.1006), version
0.240921.1006 (doi:10.48324/dandi.001193/0.240921.1006). The paper cites no version of the
dandiset (its data statement says the data are available on request; the dandiset was made two
years after publication). This is the only published version, so it is pinned. The draft holds
the same 40 files with the same checksums and sizes, and its metadata differ only in the fields
that name the version (checked below), so the choice does not change any result.

**What the dandiset holds.** One NWB file per experimental block: 10 participants, 4 blocks each.
Each file holds a single 256-channel high-gamma series and an electrode table. It holds no trial
or event times, no condition labels, no button responses, no electrode locations (anatomical
labels or MNI coordinates), no broadband signal or other frequency band, and no per-participant
sex or age. Every claim except the participant counts needs at least one of these, so all but
one claim are blocked as `data_missing`.

| Claim | Tested here | Why not |
|---|---|---|
| participants | yes | |
| task-structure | no | no trial or word event times, no condition labels |
| behaviour-response-time | no | no button responses or question onsets |
| fig1b-high-gamma-best-band | no | only high gamma; no other band, no broadband; no events |
| fig1c-responsive-in-ifg | no | no events (rest against stimulus), no electrode locations |
| fig1d-only-after-keyword | no | no events or condition labels |
| fig2b-violation-electrodes | no | no events or condition labels, no electrode locations |
| fig2b-single-type | no | depends on fig2b-violation-electrodes |
| suppfig6a-half-split | no | no events, labels or sentence identities |
| suppfig6b-duration-metric | no | no events or condition labels |
| fig2c-euclidean-similarity | no | depends on fig2b-violation-electrodes |
| suppfig7a-com-overlap | no | no COM labels, no events |
| suppfig7b-not-nonlinguistic | no | no NL labels, no events |
| fig2de-no-spatial-separation | no | no MNI coordinates; depends on fig2b-violation-electrodes |
| suppfig8b-individual-no-separation | no | no coordinates; depends on fig2b-violation-electrodes |
| fig2f-circuit-diameter | no | depends on fig2b-single-type |
| suppfig9-predict-from-corr | no | no CORR labels or events; depends on fig2b-single-type |
| fig3-latency-values | no | no keyword onsets; depends on fig2b-violation-electrodes |
| fig3d-latency-differences | no | depends on fig3-latency-values |
| fig3-sync-latency-variability | no | depends on fig3-latency-values |
| fig3e-latency-not-location | no | no coordinates; depends on fig3-latency-values |
| suppfig10a-separable-400-700 | no | no condition labels or keyword onsets |
| suppfig10b-duration-differs | no | depends on fig2b-violation-electrodes |
| nl-low-frequency | no | no 4 to 10 Hz signal, no NL labels |
| stimuli-word-frequency | no | needs only the paper's stimulus list and an unpinned web corpus; no dataset data |

In [ ]:
import time
import json

import numpy as np
import pandas as pd

import ifg_hga

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

started = time.time()
versions = ifg_hga.published_versions()
meta, files = ifg_hga.version_metadata()
draft_meta, draft_files = ifg_hga.version_metadata("draft")
summaries = ifg_hga.load_all()
print(f"dandiset {ifg_hga.DANDISET_ID}, version {ifg_hga.DANDISET_VERSION}: read {len(summaries)} files "
      f"in {time.time() - started:.0f} s")

## Data check

Each statement about the metadata is checked against the values under the output it rests on.

In [ ]:
print("Versions the archive lists:", versions)
print("Files at the pinned version:", len(files), "; in the draft:", len(draft_files))
print("Draft files identical to the pinned version (path, checksum, size):", files == draft_files)
skip = {"id", "version", "doi", "url", "citation", "manifestLocation", "dateModified", "datePublished",
        "publishedBy", "assetsSummary"}
differ = sorted(k for k in set(meta) | set(draft_meta) if k not in skip and meta.get(k) != draft_meta.get(k))
print("Other metadata fields that differ between the pinned version and the draft:", differ)
print("\nDescription:", meta["description"])
summary = meta["assetsSummary"]
print("\nArchive summary: subjects", summary.get("numberOfSubjects"), "; files", summary.get("numberOfFiles"),
      "; bytes", summary.get("numberOfBytes"), "; variables", summary.get("variableMeasured"))
print("Related resources in the metadata:", meta.get("relatedResource"))

In [ ]:
info = pd.DataFrame([{k: s[k] for k in ("path", "identifier", "subject_id", "session_start", "rate",
                                         "n_samples", "n_channels", "unit", "sex", "age")} for s in summaries])
info["block"] = info.identifier.str.split("_block").str[1].astype(int)
info["session_date"] = info.session_start.str[:10]
print("Files per subject and block (identifier field):")
print(pd.crosstab(info.subject_id, info.block).to_string())
print("\nDistinct subjects:", info.subject_id.nunique(), "; files:", len(info))
print("Each file's identifier names its subject:", bool((info.identifier.str.split("_block").str[0] == info.subject_id).all()))
print("One session date per subject:", bool(info.groupby("subject_id").session_date.nunique().eq(1).all()))
print("\nPer-subject metadata (identical in every file?)")
for key in ("sex", "age", "subject_description", "species", "session_description", "experiment_description",
            "related_publications", "series_description", "unit", "conversion", "rate", "starting_time"):
    values = sorted({json.dumps(s[key], ensure_ascii=False) for s in summaries})
    print(f"  {key}: {', '.join(values)}")
print("\nSubjects by session date:")
print(info.groupby("subject_id").session_date.first().sort_values().to_string())

The files hold 10 subjects with four blocks each, as the Methods' four blocks per participant.
Every file gives sex `U` and age `P19Y/P50Y` (the age range of the whole group), with the
subject description "chinese, 4 males and 6 females" in every file. Supplementary Table 1 gives
each participant's sex and age, but nothing in the dandiset maps the subject ids (TC5 to TC20)
to the participants numbered 1 to 10 there, so per-participant sex and age cannot be read or
checked.

In [ ]:
print("Objects outside acquisition and general, per file:", sorted({json.dumps(s["other_objects"]) for s in summaries}))
print("Files with a trials table:", sum(s["has_trials"] for s in summaries))
print("Electrode table columns:", sorted({json.dumps(s["electrode_columns"]) for s in summaries}))
print("Values of `location`:", sorted({v for s in summaries for v in s["locations"]}))
print("Values of `filtering`:", sorted({v for s in summaries for v in s["filtering"]}))
print("Electrodes per group:", sorted({json.dumps({str(k): v for k, v in s["group_counts"].items()}) for s in summaries}))
print("Group descriptions:", sorted({json.dumps(s["group_descriptions"]) for s in summaries}))
print("Group locations:", sorted({json.dumps(s["group_locations"]) for s in summaries}))
print("Series rows 0..255 in order in every file:", all(s["series_electrodes"] == list(range(256)) for s in summaries))

Apart from the high-gamma series, every file holds only empty `stimulus`, `analysis` and
`processing` groups: no trials table, no intervals, no events, no photodiode, no responses. The
electrode table has no coordinates, every `location` is `unknown`, and neither grid says which
cortex it covers (Supplementary Table 1: one grid over the IFG, the other over the MFG or STG).
The analyses of the paper start from keyword onsets, condition labels and IFG labels, none of
which is here.

In [ ]:
vals = pd.DataFrame([{k: s[k] for k in ("identifier", "n_nan", "channel_mean_absmax", "channel_std_min",
                                         "channel_std_max", "min", "p01", "p50", "p99", "max", "frac_above_10",
                                         "first_sample_median", "start_transient_samples", "n_duplicate_channels")} for s in summaries])
print("Values per file (z units if the data are z scores):")
print(vals.round(4).to_string(index=False))
print("\nLargest |channel mean| over all files:", f"{vals.channel_mean_absmax.max():.2e}")
print("Channel standard deviations over all files:", round(vals.channel_std_min.min(), 5), "to", round(vals.channel_std_max.max(), 5))
print("NaN values:", int(vals.n_nan.sum()), "; identical channel pairs within a file:", int(vals.n_duplicate_channels.sum()))
starts = vals[vals.first_sample_median >= 3]
print("Files whose first sample has a channel median of 3 z or more:", len(starts), sorted(set(starts.identifier.str.split("_").str[0])))
print("  samples until the channel median falls below 3 z:", int(starts.start_transient_samples.min()), "to",
      int(starts.start_transient_samples.max()))

The values are stated in volts, but every channel of every file has mean 0 (largest absolute
mean printed above) and standard deviation 1 to within 3e-5. They are z scores per channel and
per file, consistent with the Methods' HGA "z scored relative to the mean and standard deviation
of the data within each experimental block", and with each file being one block. The unit
`volts` is wrong. Because each file is z scored on its own, no channel can be identified as a bad
channel from its scale, and the dandiset does not say which channels or intervals were excluded
(the Methods exclude both); all 256 channels are present in every file. Some files (counted
above; all those of TC5, TC6, TC7 and TC9) begin with a transient shared by all channels, with a
channel median of up to 50 z at the first sample, which falls below 3 z within the few samples
printed above.

In [ ]:
rate_check = info.groupby("subject_id").agg(blocks=("block", "size"), samples_min=("n_samples", "min"),
                                             samples_max=("n_samples", "max"))
for hz in (400.0, ifg_hga.PAPER_HGA_HZ):
    rate_check[f"block_s_at_{int(hz)}Hz_min"] = rate_check.samples_min / hz
    rate_check[f"block_s_at_{int(hz)}Hz_max"] = rate_check.samples_max / hz
    rate_check[f"s_per_trial_at_{int(hz)}Hz_max"] = rate_check.samples_max / hz / ifg_hga.MIN_LINGUISTIC_TRIALS_PER_BLOCK
need = ifg_hga.MIN_LINGUISTIC_TRIALS_PER_BLOCK * ifg_hga.MIN_TRIAL_S
print(f"A block holds at least {ifg_hga.MIN_LINGUISTIC_TRIALS_PER_BLOCK} linguistic trials of at least "
      f"{ifg_hga.MIN_TRIAL_S:.1f} s before the question: at least {need:.0f} s.")
print(rate_check.round(1).to_string())
print(f"\nBlocks shorter than {need:.0f} s at the stated 400 Hz:", int((info.n_samples / 400 < need).sum()), "of", len(info))
print(f"Blocks shorter than {need:.0f} s at the paper's {ifg_hga.PAPER_HGA_HZ:.0f} Hz:",
      int((info.n_samples / ifg_hga.PAPER_HGA_HZ < need).sum()), "of", len(info))

excess = pd.DataFrame([{"identifier": s["identifier"], "word_rate_if_400Hz": s["excess_word_rate_if_400"],
                        "word_rate_if_100Hz": s["excess_word_rate_if_100"]} for s in summaries])
print("\nPower at the 2 Hz word rate over the flanking band (log10, mean over channels), under each rate:")
print(excess.describe().loc[["mean", "min", "50%", "max"]].round(3).to_string())
print("Files with excess above 0.1 (a 26% peak):", {c: int((excess[c] > 0.1).sum()) for c in excess.columns[1:]})
print("Files with any excess above 0:", {c: int((excess[c] > 0).sum()) for c in excess.columns[1:]}, "of", len(excess))

The series are stated at 400 Hz. At that rate the blocks last 70 to 95 s, under 3 s per
linguistic trial, while the task needs at least 4.3 s per trial before the question; every block
is too short (count above). At 100 Hz, the rate the Methods give for the HGA, the blocks last 282
to 381 s, 8.8 to 11.9 s per trial, which the task allows. The data are consistent with a rate of
100 Hz, not 400 Hz, unless each file holds less than half of its block (the Methods exclude bad
intervals, but say nothing of that size). The 400 Hz is the rate the Methods give for the signal
before the HGA was taken. The spectral check at the 2 Hz word rate leans the same way but is
weak: at the period the 100 Hz reading predicts, the power is above its flanking band in 34 of 40
files (median excess 0.021 log10, about 5%), against 19 of 40 (median -0.001) at the period the
400 Hz reading predicts; only one file shows a peak above 0.1 under either reading. The blocks'
length is the stronger argument. Neither check shows the rate directly. Without event times, no stream can be shown to share a clock with a trial table: this
could not be established for any session.

In [ ]:
names = [s["identifier"] for s in summaries]
match = np.array([[ifg_hga.best_match(a["fingerprint"], b["trace"]) if a is not b else np.nan for b in summaries]
                  for a in summaries])
pairs = [(names[i], names[j], round(match[i, j], 4)) for i in range(len(names)) for j in range(len(names))
         if i != j and match[i, j] > 0.9]
print(f"Largest correlation of {ifg_hga.FINGERPRINT_SAMPLES} samples from the middle of one file (differenced "
      "channel mean) with any stretch of another file:")
print("  over all pairs, max", round(np.nanmax(match), 4), "; median", round(np.nanmedian(match), 4))
print("  pairs above 0.9:", pairs)
i, j = np.unravel_index(np.nanargmax(match), match.shape)
print("  most similar pair:", names[i], names[j])
print("Distinct sample counts:", info.n_samples.nunique(), "of", len(info))
print("Self-match of each file (should be 1):", round(min(ifg_hga.best_match(s["fingerprint"], s["trace"]) for s in summaries), 6))

No block was deposited twice: no stretch from the middle of a file matches any stretch of
another file (largest correlation printed above, against 1 for a file with itself), and all 40
sample counts differ. Each recording is counted once.

## participants

> Ten participants (four male, six female, 19 to 50 years) were recorded with two 128-electrode
> grids each; all ten did the CORR, SYN-P, SYN-C and SEM conditions, five of them the combined
> violation (COM) and the other five the non-linguistic task (NL).

Criterion: counts of distinct subjects, their sex, and the tasks each has; tolerance zero;
reproduced when all match, partially reproduced when the dataset holds a subset with matching
metadata; ages compared with Supplementary Table 1 where the dataset gives them.

In [ ]:
per_subject = pd.DataFrame([{"subject_id": s["subject_id"], "sex": s["sex"], "age": s["age"],
                             "electrodes": s["n_channels"], "grid_1": s["group_counts"].get("Grid_1"),
                             "grid_2": s["group_counts"].get("Grid_2")} for s in summaries]).drop_duplicates()
print(per_subject.sort_values("subject_id").to_string(index=False))
n_subjects = per_subject.subject_id.nunique()
males = int((per_subject.sex == "M").sum())
females = int((per_subject.sex == "F").sum())
print("\nExpected against found:")
rows = [
    ("participants", 10, n_subjects),
    ("male participants (sex field)", 4, males),
    ("female participants (sex field)", 6, females),
    ("participants with COM", 5, "no task or condition labels in the files"),
    ("participants with NL", 5, "no task or condition labels in the files"),
    ("electrodes per participant", 256, sorted(per_subject.electrodes.unique().tolist())),
]
for what, expected, found in rows:
    print(f"  {what}: expected {expected}, found {found}")
print("  sex as free text in every file's subject description:", sorted({s["subject_description"] for s in summaries}))
print("  grids per participant: 2 of 128 in every file:",
      bool(((per_subject.grid_1 == 128) & (per_subject.grid_2 == 128)).all()))
print("  ages: every file gives", sorted(per_subject.age.unique()), "; Supplementary Table 1 range is 19 to 50 years")

**Result: partially reproduced.** The dataset holds 10 distinct participants, each with 256
electrodes in two groups of 128, as the paper states. The sex field is `U` for all 10, so the
counts of 4 male and 6 female cannot be found from the subjects (0 and 0 are found); the 4 and 6
appear only as a sentence copied into every file's subject description. Ages are given only as
the group's range (19 to 50 years), which matches the paper's range but not per participant. No
file says which task a participant did, so the 5 COM and 5 NL participants cannot be counted.
Two of the six counts match from the data; the others are not in it.

## Summary

| Claim | Outcome |
|---|---|
| participants | partially reproduced |
| the other 24 claims | blocked, data missing (stimuli-word-frequency: blocked, other) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** Only the counts of participants and electrodes
are tested. Every analysis of the paper aligns the high-gamma activity to keyword onsets and
compares conditions within IFG electrodes; the dataset holds none of the event times, condition
labels or electrode locations these need.

**What was found about the dataset.**

- No trial, word or keyword event times, condition labels, button responses or photodiode signal.
- No electrode locations: every `location` is `unknown`, no coordinates, and the two grids do not
  say which cortex each covers.
- Only high gamma, so the frequency-band analyses (Fig. 1b, Supplementary Figs. 3, 4, 7c) cannot
  be repeated.
- Sex `U` and age `P19Y/P50Y` for every subject, and no mapping from the subject ids to the
  participants of Supplementary Table 1.
- The unit `volts` is wrong: the values are z scores per channel and block.
- The stated rate of 400 Hz is inconsistent with the length of the blocks and consistent with
  the paper's 100 Hz HGA.
- No record of the bad channels and intervals the Methods exclude.
- No recording deposited twice.